## 4. Evaluation

### 4.1 Importing the Library

In [19]:
import pickle
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.models import load_model

In [20]:
DATA_DIR = "../dataset/processed"
MODEL_DIR = "notebook/models"

### 4.2 Load the Data

In [21]:
X_test_tfidf  = pickle.load(open(os.path.join(DATA_DIR, "X_test_tfidf.pkl"), "rb"))
X_test_pad    = pickle.load(open(os.path.join(DATA_DIR, "X_test_pad.pkl"), "rb"))
y_test        = pickle.load(open(os.path.join(DATA_DIR, "y_test.pkl"), "rb"))

### 4.3 Check Array Dimension

In [22]:
y_test_sklearn = y_test.astype(int).ravel()

### 4.4 Load Models

In [23]:
rf_model     = pickle.load(open(os.path.join(MODEL_DIR, "rf_model.pkl"), "rb"))
lstm_model   = load_model(os.path.join(MODEL_DIR, "lstm_model.keras"))
bilstm_model = load_model(os.path.join(MODEL_DIR, "bilstm_model.keras"))

FileNotFoundError: [Errno 2] No such file or directory: 'notebook/models/rf_model.pkl'

### 4.5 Evaluating Function

In [ ]:
def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    print(f"Akurasi : {acc:.4f}")
    # y_true di sini harus y_test_sklearn
    print(classification_report(y_true, y_pred))
    return acc

### 4.6 Predicting and Evaluate All Models

In [ ]:
scores = {}

# rf x tfidf
rf_pred = rf_model.predict(X_test_tfidf)
scores['Random Forest (TF-IDF)'] = evaluate_model("Random Forest (TF-IDF)", y_test_sklearn, rf_pred)


# lstm x tfidf
lstm_pred_prob = lstm_model.predict([X_test_pad, X_test_tfidf])
lstm_pred = (lstm_pred_prob > 0.5).astype(int).ravel() 
scores['LSTM + TF-IDF'] = evaluate_model("LSTM + TF-IDF", y_test_sklearn, lstm_pred)


# bilstm x tfidf
bilstm_pred_prob = bilstm_model.predict([X_test_pad, X_test_tfidf])
bilstm_pred = (bilstm_pred_prob > 0.5).astype(int).ravel()
scores['BiLSTM + TF-IDF'] = evaluate_model("BiLSTM + TF-IDF", y_test_sklearn, bilstm_pred)

### 4.7 Visualizing

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(scores.keys(), scores.values(), color=['skyblue', 'salmon', 'lightgreen'])
plt.xticks(rotation=10) # Rotasi sedikit saja cukup
plt.ylabel("Accuracy")
plt.title("Perbandingan Akurasi Model")
plt.ylim(0, 1.0) # Set batas Y dari 0.0 sampai 1.0

# Tambahkan label angka di atas bar
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, yval + 0.01, f'{yval:.4f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()